# Chatbot Demo

In [1]:
import sys
!{sys.executable} -m pip install python-dotenv

In [2]:
import os, sys

BASE = '/Users/kaylatom/Desktop/DUKE STUFF/CS 372/cs372_final_project'
os.chdir(BASE)
sys.path.append(os.path.join(BASE, 'src'))

from dotenv import load_dotenv
load_dotenv()
print('Ready.')

Ready.


In [3]:
import pickle, numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

with open('data/embeddings.pkl', 'rb') as f:
    store = pickle.load(f)

embeddings = store['embeddings']
metadata   = store['metadata']
model      = SentenceTransformer(store['model_name'])
print(f'Loaded {len(metadata)} menu items.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded 394 menu items.


In [4]:
from groq import Groq
client = Groq(api_key=os.environ['GROQ_API_KEY'])
MODEL  = 'llama-3.3-70b-versatile'

SYSTEM = """You are Duke Bites, a friendly dining assistant for Duke University students.
Help students decide what to eat based on their mood or craving.
You will be given relevant menu items from Duke dining halls as context.
Always recommend 2-3 specific items, include the dining location and hours.
Be warm and conversational. Keep responses to 3-5 sentences.

Retrieved menu items:
{context}"""

def retrieve(query, top_k=5):
    query_vec = model.encode([query])
    scores    = cosine_similarity(query_vec, embeddings)[0]
    top_idx   = np.argsort(scores)[::-1][:top_k]
    results   = []
    for idx in top_idx:
        row = metadata.iloc[idx]
        results.append({
            'item':        row.get('name_item',        row.get('name', '')),
            'location':    row.get('name_location',    ''),
            'description': row.get('description_item', row.get('description', '')),
            'tags':        row.get('generated_tags',   ''),
            'meal_period': row.get('meal_period',      ''),
            'hours':       row.get('hours',            ''),
            'score':       round(float(scores[idx]),   3),
        })
    return results

def format_context(results):
    lines = []
    for r in results:
        lines.append(
            f"- {r['item']} @ {r['location']} ({r['meal_period']})\n"
            f"  Description: {r['description']}\n"
            f"  Tags: {r['tags']}\n"
            f"  Hours: {r['hours']}"
        )
    return '\n'.join(lines)

def chat(user_message, history):
    results = retrieve(user_message, top_k=5)
    context = format_context(results)
    system  = SYSTEM.format(context=context)
    messages = (
        [{'role': 'system', 'content': system}]
        + history
        + [{'role': 'user', 'content': user_message}]
    )
    resp = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=0.7, max_tokens=400
    )
    reply = resp.choices[0].message.content
    history = history + [
        {'role': 'user',      'content': user_message},
        {'role': 'assistant', 'content': reply},
    ]
    return reply, history

print('Chatbot ready.')

Chatbot ready.


In [5]:
# --- Chat here ---
# Edit the messages list and re-run this cell to keep chatting

history = []

messages = [
    "I'm really hungry and want something comforting",
    "Actually I'm vegetarian, any options?",
    "What time does that place close?",
]

for msg in messages:
    print(f"You: {msg}")
    reply, history = chat(msg, history)
    print(f"Duke Bites: {reply}\n")

You: I'm really hungry and want something comforting
Duke Bites: I've got just the thing for you. Head over to Gothic Grill, open from 11 am to Midnight, and try their Crispy Cauliflower or Mozzarella Sticks - both are comfort food faves that are sure to hit the spot. Alternatively, you could also swing by the Cafe, open from 7 am to 10 pm, and grab a Crispy Chicken and Pimento Cheese Sandwich for a satisfying and filling meal.

You: Actually I'm vegetarian, any options?
Duke Bites: In that case, I'd recommend checking out The Skillet, open from 7 am to 9 pm, and trying their Vegetarian Omelet Combo - it's a comforting and delicious option. Alternatively, you could head to Tandoor, open from 11 am to 9 pm, and try their Specialty vegetarian combo or explore their Vegetable Options for a satisfying and savory meal.

You: What time does that place close?
Duke Bites: The Skillet is open from 7 am to 9 pm, so you've got plenty of time to grab a bite. If you're looking for something comfort

In [6]:
# --- Single message cell ---
# Use this to send one message at a time and keep history going

user_input = "I want something spicy for dinner"  # <- change this

reply, history = chat(user_input, history)
print(f"You: {user_input}")
print(f"Duke Bites: {reply}")

You: I want something spicy for dinner
Duke Bites: Spicy food is just what you need to liven up dinner. I'd recommend heading to Ginger + Soy, open from 11 am to 9 pm, and trying their Spicy Miso Ramen - it's a flavorful and spicy bowl of goodness. Alternatively, you could also try the Spicy Cauliflower Wrap at Sprout, open from 11 am to 9 pm, for a vegetarian option that packs a punch. If you're in the mood for something different, Il Forno's Spicy Il Forno, also open from 11 am to 9 pm, is a great choice.
